# Build Your Own GPT

## Project Goal

In this project, we will build a GPT-style language model from scratch using PyTorch.

The model will:

- Load a text corpus
- Tokenize text into numerical representations
- Build training batches
- Implement self-attention
- Train a Transformer model
- Generate novel text

Dataset:
- Tiny Shakespeare

Framework:
- PyTorch

Hardware:
- CPU

# 1. Imports and Configuration

This section imports the required libraries and defines the hyperparameters used throughout the project.

Key hyperparameters:

- batch_size: Number of sequences processed simultaneously
- block_size: Context window length
- n_embd: Embedding dimension
- n_head: Number of attention heads
- n_layer: Number of Transformer blocks

In [282]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

batch_size = 16
block_size = 32

max_iters = 5000
eval_interval = 100

learning_rate = 1e-3

device = "cpu"

eval_iters = 200

n_embd = 64
n_head = 4
n_layer = 4

dropout = 0.2

# 2. Load the Dataset

The Tiny Shakespeare dataset contains approximately 1.1 million characters.

We load the entire corpus into memory and inspect its structure before preprocessing and tokenization.

In [283]:
with open("../data/tiny_shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(len(text))
print(text[:500])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


# 3. Build the Vocabulary

A vocabulary is the set of unique characters present in our text corpus.

Each character will later be assigned a numerical index.

Examples:

a → 0

b → 1

c → 2

In [284]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


# 4. Character Tokenization

Neural networks cannot process text directly.

We therefore create two mappings:

- stoi (string to index)
- itos (index to string)

We also create helper functions:

- encode()
- decode()


In [285]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello"))
print(decode(encode("hello")))


[46, 43, 50, 50, 53]
hello


# 5. Encode the Dataset

Using the encode function, convert every character in the training corpus into an integer.

The resulting tensor will form the foundation of our training data.

In [286]:
data = torch.tensor(
    encode(text),
    dtype=torch.long
)

print(data.shape)
print(data[:100])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


# 6. Training and Validation Split

To evaluate generalization performance, we reserve:

- 90% of data for training
- 10% of data for validation

In [287]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print(len(train_data))
print(len(val_data))


1003854
111540


# 7. Batch Generation

Training occurs on small chunks of text called batches.

Input sequences (x):

hello wo

Target sequences (y):

ello wor

The model learns to predict the next character at every position.

# 8. Context Windows

Large Language Models do not process an entire document at once.

Instead, they work with a limited amount of previous text called a **context window**.

For example, if the context window contains:

"to be or not"

the model attempts to predict:

"to"

The `block_size` hyperparameter defines the maximum number of previous tokens the model can use when predicting the next token.

In this project:

- `block_size = 32`
- Each training example contains 32 input tokens
- The target is the next token at every position

The model learns patterns by repeatedly seeing input sequences and their shifted target sequences.

In [288]:
block_size = 32

x = train_data[:block_size]
y = train_data[1:block_size + 1]

print("Input:")
print(x)

print("\nTarget:")
print(y)

Input:
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42])

Target:
tensor([47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44, 53,
        56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1])


## Understanding Next-Token Prediction

The target sequence is simply the input sequence shifted by one position.

Example:

Input:

h e l l o

Target:

e l l o _

During training, the model learns:

- h → e
- he → l
- hel → l
- hell → o

This next-token prediction objective is the foundation of GPT training.

In [289]:
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]

    print(
        f"When input is {context.tolist()} "
        f"the target is {target.item()}"
    )

When input is [18] the target is 47
When input is [18, 47] the target is 56
When input is [18, 47, 56] the target is 57
When input is [18, 47, 56, 57] the target is 58
When input is [18, 47, 56, 57, 58] the target is 1
When input is [18, 47, 56, 57, 58, 1] the target is 15
When input is [18, 47, 56, 57, 58, 1, 15] the target is 47
When input is [18, 47, 56, 57, 58, 1, 15, 47] the target is 58
When input is [18, 47, 56, 57, 58, 1, 15, 47, 58] the target is 47
When input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47] the target is 64
When input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64] the target is 43
When input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43] the target is 52
When input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52] the target is 10
When input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10] the target is 0
When input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0] the target is 14
When input is [18, 47, 56, 57, 58, 1, 15, 47, 58, 47,

## Creating Training Batches

Training on a single sequence at a time would be inefficient.

Instead, we process multiple sequences in parallel using batches.

The input tensor shape is:

(batch_size, block_size)

where:

- batch_size = number of sequences
- block_size = context length

The target tensor has the same shape.


In [290]:
def get_batch(split):

    data = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack(
        [data[i:i + block_size] for i in ix]
    )

    y = torch.stack(
        [data[i + 1:i + block_size + 1] for i in ix]
    )

    return x, y

## Inspecting a Batch

The batch generator randomly selects starting positions from the dataset.

Each row represents an independent training example.

This helps the model learn from multiple contexts simultaneously and greatly improves training efficiency.

In [291]:
xb, yb = get_batch("train")

print("Input Batch Shape:", xb.shape)
print("Target Batch Shape:", yb.shape)

print("\nFirst Input Sequence:")
print(xb[0])

print("\nFirst Target Sequence:")
print(yb[0])

Input Batch Shape: torch.Size([16, 32])
Target Batch Shape: torch.Size([16, 32])

First Input Sequence:
tensor([58, 53,  1, 41, 53, 56, 56, 59, 54, 58,  1, 39,  1, 51, 39, 52,  5, 57,
         1, 61, 47, 44, 43,  1, 47, 57,  0, 61, 46, 43, 52,  1])

First Target Sequence:
tensor([53,  1, 41, 53, 56, 56, 59, 54, 58,  1, 39,  1, 51, 39, 52,  5, 57,  1,
        61, 47, 44, 43,  1, 47, 57,  0, 61, 46, 43, 52,  1, 57])


# 9. Token Embeddings

Neural networks cannot work directly with integer token IDs.

For example:

```text
h → 17
e → 9
l → 21
```

The model does not understand that these numbers represent characters.

Instead, each token is mapped to a learned vector called an **embedding**.

Rather than representing a token as a single integer, it becomes a dense vector of floating-point values.

Example:

```text
h → [0.23, -0.51, 1.12, ...]
```

These embeddings are learned during training and allow the model to capture relationships between different tokens.

The embedding dimension is controlled by:

```python
n_embd = 64
```

meaning every token is represented by a vector containing 64 learned values.

In [292]:
# Create the token embedding table

token_embedding_table = nn.Embedding(
    vocab_size,
    n_embd
)

print(token_embedding_table)

Embedding(65, 64)


## Understanding the Embedding Layer

The embedding layer contains:

```text
vocab_size × n_embd
```

learnable parameters.

For our dataset:

```python
vocab_size = 65
n_embd = 64
```

The embedding table shape becomes:

```text
65 × 64
```

Each row corresponds to one token in the vocabulary.

In [293]:
print("Embedding table shape:")
print(token_embedding_table.weight.shape)

Embedding table shape:
torch.Size([65, 64])


## Converting Tokens into Embeddings

Let's retrieve a batch and pass the token IDs through the embedding layer.

Input tensor shape:

```text
(B, T)
```

where:

- B = batch size
- T = context length

Output shape:

```text
(B, T, C)
```

where:

- C = embedding dimension

In [294]:
xb, yb = get_batch("train")

token_embeddings = token_embedding_table(xb)

print("Input shape:")
print(xb.shape)

print("\nEmbedding shape:")
print(token_embeddings.shape)

Input shape:
torch.Size([16, 32])

Embedding shape:
torch.Size([16, 32, 64])


## Examine One Token Embedding

Each token now has a 64-dimensional representation.

The values are initially random and will gradually learn meaningful patterns during training.


In [295]:
print("First token ID:")
print(xb[0, 0])

print("\nEmbedding vector:")
print(token_embeddings[0, 0])

print("\nEmbedding length:")
print(len(token_embeddings[0, 0]))

First token ID:
tensor(59)

Embedding vector:
tensor([-1.0889,  1.0108,  0.8543, -0.0441,  1.8017,  0.6014, -2.5448, -0.4865,
         2.6412,  1.6053,  0.5901,  0.8137, -0.1124, -0.3050,  1.1426,  0.6637,
        -0.7000,  0.9262, -1.1032, -1.2125,  0.6065,  0.5882, -0.5453,  0.7654,
         0.5692,  0.8859, -0.0700,  0.6792, -0.0283, -1.2243, -1.7192,  1.4801,
         0.9587, -0.0338,  0.5083, -0.2502,  2.0734, -0.2994,  0.0473, -0.9626,
         1.3064, -0.2256, -1.8305, -0.4002,  0.3302,  1.5454,  1.3778, -1.2796,
        -0.6325, -1.3849,  0.7045, -1.6437,  1.0897, -1.6458, -0.1595, -0.2041,
         0.7659,  1.6846,  2.2841,  1.8561,  0.8824,  1.0156,  0.1238,  0.4307],
       grad_fn=<SelectBackward0>)

Embedding length:
64


## Why Embeddings Matter

Without embeddings:

```text
a = 0
b = 1
c = 2
```

The model incorrectly assumes numerical relationships exist between tokens.

With embeddings:

```text
a → vector
b → vector
c → vector
```

The model learns similarities and relationships automatically.

Embeddings form the foundation of modern NLP systems and are one of the key breakthroughs that enabled Transformer architectures.

In [296]:
num_parameters = token_embedding_table.weight.numel()

print(f"Embedding parameters: {num_parameters:,}")

Embedding parameters: 4,160


# 10. Positional Embeddings

Token embeddings tell the model **what** a token is, but they do not tell the model **where** a token appears in a sequence.

Consider these two sentences:

```text
dog bites man
```

```text
man bites dog
```

Both sentences contain the same words, but the order completely changes the meaning.

Transformers have no built-in understanding of sequence order, so we must explicitly provide positional information.

GPT accomplishes this using **learnable positional embeddings**.

Each position in the context window receives its own embedding vector:

```text
Position 0 → vector
Position 1 → vector
Position 2 → vector
...
Position 31 → vector
```

The final representation for each token is:

```text
Token Embedding + Positional Embedding
```

This gives the model information about both:

- What the token is
- Where the token appears

In [297]:
# Create positional embedding table

position_embedding_table = nn.Embedding(
    block_size,
    n_embd
)

print(position_embedding_table)

Embedding(32, 64)


## Positional Embedding Shape

Unlike token embeddings, positional embeddings depend on position rather than vocabulary.

Shape:

```python
(block_size, n_embd)
```

For example:

```python
(32, 64)
```

This creates one learnable embedding for every position in the context window.


In [298]:
print("Positional embedding shape:")
print(position_embedding_table.weight.shape)

Positional embedding shape:
torch.Size([32, 64])


## Generate Position Indices

For a context window of length 32:

```text
0
1
2
3
...
31
```

Each index represents a position in the sequence.

In [299]:
positions = torch.arange(block_size)

print(positions)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])


## Look Up Positional Embeddings

Each position index is converted into a learnable embedding vector.

Resulting shape:

```text
(block_size, n_embd)
```

In [300]:
position_embeddings = position_embedding_table(positions)

print(position_embeddings.shape)

torch.Size([32, 64])


## Combining Token and Positional Information

We now retrieve a batch and create:

- Token embeddings
- Positional embeddings

The two tensors are then added together.

This combined representation is what enters the Transformer.

In [301]:
xb, yb = get_batch("train")

token_embeddings = token_embedding_table(xb)

position_embeddings = position_embedding_table(
    torch.arange(xb.shape[1])
)

print("Token Embeddings Shape:")
print(token_embeddings.shape)

print("\nPosition Embeddings Shape:")
print(position_embeddings.shape)

Token Embeddings Shape:
torch.Size([16, 32, 64])

Position Embeddings Shape:
torch.Size([32, 64])


## Create the Input Representation

PyTorch automatically broadcasts the positional embeddings across the batch dimension.

Shapes:

Token embeddings:

```text
(B, T, C)
```

Position embeddings:

```text
(T, C)
```

Result:

```text
(B, T, C)
```

In [302]:
x = token_embeddings + position_embeddings

print("Combined embedding shape:")
print(x.shape)


Combined embedding shape:
torch.Size([16, 32, 64])


## Inspect the Combined Representation

Each token now contains:

1. Semantic information (token embedding)
2. Location information (positional embedding)

This representation becomes the input to the self-attention mechanism.

In [303]:
print("Combined embedding for first token:")
print(x[0, 0])

print("\nEmbedding length:")
print(len(x[0, 0]))


Combined embedding for first token:
tensor([-0.6328, -1.1083, -2.1483,  0.6591,  0.8919,  1.7355, -0.8867, -0.8814,
         0.8059, -0.0241, -0.2995,  1.0164, -0.4513, -0.3673, -1.0664, -0.7261,
        -1.0062,  0.1003,  0.8769,  0.8174, -1.6650, -0.4216,  0.4114, -3.1248,
        -0.6256, -0.8810,  2.9073, -1.7287, -0.5076,  1.4377,  1.7887, -0.4410,
        -1.3880,  2.2976, -0.5645, -0.2855, -1.8738,  1.6776, -1.8654, -0.4058,
        -1.1389, -0.7219,  0.2611,  0.1774,  0.8216,  0.3088,  0.0923,  0.7147,
         0.8362,  0.6577,  0.1210,  0.8171,  0.1057,  0.4888, -1.5136, -1.0267,
         0.9682, -1.4820,  0.6745, -1.0517,  0.5291,  2.1008,  0.8706, -0.2070],
       grad_fn=<SelectBackward0>)

Embedding length:
64


## Why Positional Embeddings Matter

Without positional embeddings:

```text
dog bites man
```

and

```text
man bites dog
```

would appear nearly identical to the model.

Positional embeddings allow the Transformer to understand sequence order and grammatical structure.

They are one of the key components that made GPT-style architectures successful.

# 11. Self-Attention

Self-attention is the mechanism that made Transformers revolutionary.

Instead of processing tokens sequentially like an RNN, self-attention allows every token to examine previous tokens and decide which ones are important.

Example:

```text
The king ruled his kingdom wisely.
```

When processing:

```text
kingdom
```

the model may pay attention to:

```text
king
ruled
his
```

Self-attention creates dynamic connections between tokens, allowing the model to build rich contextual representations.

GPT uses **masked self-attention**, meaning a token can only attend to itself and previous tokens, never future tokens.

## Understanding Queries, Keys, and Values

Every token embedding is transformed into three vectors:

### Query (Q)

What information am I looking for?

### Key (K)

What information do I contain?

### Value (V)

What information should I contribute?

The model compares queries against keys to determine attention strength.

Attention scores are then applied to values.

In [304]:
head_size = 16

key = nn.Linear(n_embd, head_size, bias=False)
query = nn.Linear(n_embd, head_size, bias=False)
value = nn.Linear(n_embd, head_size, bias=False)

## Create Query, Key and Value Tensors

We first generate a fresh batch and convert it into embeddings.

Shape convention:

```text
B = Batch Size
T = Context Length
C = Embedding Dimension
```

Current dimensions:

```text
(B, T, C)
```

In [305]:
xb, yb = get_batch("train")

token_embeddings = token_embedding_table(xb)

position_embeddings = position_embedding_table(
    torch.arange(xb.shape[1])
)

x = token_embeddings + position_embeddings

print(x.shape)

torch.Size([16, 32, 64])


## Compute Keys, Queries and Values

Each token is projected into three different spaces:

- Key
- Query
- Value

These projections are learnable.

In [306]:
k = key(x)
q = query(x)
v = value(x)

print("Key Shape:", k.shape)
print("Query Shape:", q.shape)
print("Value Shape:", v.shape)

Key Shape: torch.Size([16, 32, 16])
Query Shape: torch.Size([16, 32, 16])
Value Shape: torch.Size([16, 32, 16])


## Calculate Attention Scores

Attention scores measure the similarity between queries and keys.

Matrix multiplication:

```text
Q × Kᵀ
```

produces:

```text
(B, T, T)
```

This matrix describes how strongly each token attends to every other token.

In [307]:
wei = q @ k.transpose(-2, -1)

print(wei.shape)

torch.Size([16, 32, 32])


## Understanding the Attention Matrix

Each row represents:

```text
Current token
```

Each column represents:

```text
Potential token to attend to
```

Large values indicate stronger relevance.

Before masking:

```text
Future tokens are visible.
```

GPT must prevent this.

In [308]:
print(wei[0])


tensor([[ 4.0130e+00,  1.2629e+00, -1.6491e-01,  ...,  1.4299e+00,
          1.2631e+00,  5.4120e+00],
        [ 3.1233e+00,  1.6866e-01, -1.7970e+00,  ...,  5.4371e+00,
         -2.2925e+00,  5.5937e-01],
        [ 1.3820e+00, -1.8454e+00, -1.1264e+00,  ..., -1.8315e-01,
         -4.5658e+00,  2.4795e+00],
        ...,
        [ 2.8551e-01,  7.5126e-01,  1.6798e-03,  ..., -3.4783e-01,
         -2.0545e+00,  2.8650e+00],
        [ 1.3929e+00, -7.4111e-01, -3.9975e+00,  ..., -4.5380e-01,
         -3.5680e+00,  6.9223e+00],
        [-7.6623e-01, -4.2240e+00,  5.0142e-01,  ..., -2.1786e-01,
          9.8269e+00,  3.7352e-01]], grad_fn=<SelectBackward0>)


## Causal Masking

To prevent cheating during training, GPT hides future tokens.

Example:

Token 5 can attend to:

```text
0 1 2 3 4 5
```

but NOT:

```text
6 7 8 ...
```

This is implemented with a lower triangular matrix.

In [309]:
tril = torch.tril(
    torch.ones(block_size, block_size)
)

print(tril[:8, :8])

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])


## Apply the Mask

All future-token positions are replaced with:

```text
-inf
```

After applying softmax:

```text
Probability = 0
```

for all future positions.

In [310]:
wei = wei.masked_fill(
    tril == 0,
    float("-inf")
)

## Convert Scores into Probabilities

The softmax function converts raw attention scores into probabilities.

Each row now sums to:

```text
1.0
```

In [311]:
wei = F.softmax(
    wei,
    dim=-1
)

print(wei.shape)

torch.Size([16, 32, 32])


In [312]:
print(wei[0])

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [9.5048e-01, 4.9519e-02, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [8.9201e-01, 3.5378e-02, 7.2610e-02,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [9.5275e-03, 1.5179e-02, 7.1732e-03,  ..., 5.0573e-03, 0.0000e+00,
         0.0000e+00],
        [9.6950e-04, 1.1475e-04, 4.4209e-06,  ..., 1.5295e-04, 6.7930e-06,
         0.0000e+00],
        [2.4884e-05, 7.8383e-07, 8.8400e-05,  ..., 4.3060e-05, 9.9186e-01,
         7.7787e-05]], grad_fn=<SelectBackward0>)


## Compute Weighted Values

The attention probabilities are used to combine value vectors.

This produces contextualized token representations.

Formula:

```text
Attention = Softmax(QKᵀ)V
```

This is the core computation inside every Transformer.

In [313]:
out = wei @ v

print(out.shape)

torch.Size([16, 32, 16])


## Interpret the Result

The output tensor contains contextualized representations.

Each token now contains information gathered from all previous tokens it attended to.

This allows the model to understand:

- Long-range dependencies
- Context
- Grammar
- Sentence structure

Self-attention is the fundamental building block of GPT architectures.

In [314]:
print("Output shape:")
print(out.shape)

print("\nFirst token context vector:")
print(out[0, 0])

Output shape:
torch.Size([16, 32, 16])

First token context vector:
tensor([ 0.0240, -0.9525, -0.1725,  0.6408, -1.7839, -0.3933, -1.1675,  0.7827,
        -1.4765, -1.7173,  0.2927, -0.2126,  0.9081,  0.3475,  0.2692,  0.1244],
       grad_fn=<SelectBackward0>)


# 12. Single Attention Head (Head Class)

In the previous section we manually implemented self-attention:

```text
Q × Kᵀ
↓
Mask
↓
Softmax
↓
V
```

While this helped us understand the mechanics, we don't want to rewrite this code every time we need an attention layer.

Instead, we encapsulate the logic inside a reusable PyTorch module called:

```python
Head
```

A single attention head learns one particular way of interpreting relationships between tokens.

Later, multiple heads will work together to form:

```python
MultiHeadAttention
```

which is a core component of GPT.

In [315]:
class Head(nn.Module):
    """
    Single self-attention head
    """

    def __init__(self, head_size):

        super().__init__()

        self.key = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(block_size, block_size)
            )
        )

        self.dropout = nn.Dropout(dropout)

## Understanding the Components

### Key Projection

```python
self.key
```

Learns what information each token contains.

### Query Projection

```python
self.query
```

Learns what information each token is searching for.

### Value Projection

```python
self.value
```

Stores information that may be passed to other tokens.

### Causal Mask

```python
self.tril
```

Ensures tokens cannot view future positions.

### Dropout

```python
self.dropout
```

Acts as a regularization mechanism and helps reduce overfitting.

In [316]:
def forward(self, x):

    B, T, C = x.shape

    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2, -1)

    wei = wei * (k.shape[-1] ** -0.5)

    wei = wei.masked_fill(
        self.tril[:T, :T] == 0,
        float("-inf")
    )

    wei = F.softmax(
        wei,
        dim=-1
    )

    wei = self.dropout(wei)

    v = self.value(x)

    out = wei @ v

    return out


Head.forward = forward

## Why Do We Scale the Attention Scores?

The line:

```python
wei = wei * (k.shape[-1] ** -0.5)
```

implements:

```text
1 / √dk
```

from the original Transformer paper.

Without scaling:

- attention scores become very large
- softmax becomes unstable
- gradients can become problematic

Scaling improves numerical stability and training performance.

## Test the Head Module

Now we instantiate a single attention head and verify that it produces correctly shaped outputs.

Input shape:

```text
(B, T, C)
```

Expected output shape:

```text
(B, T, head_size)
```

In [317]:
head_size = 16

head = Head(head_size)

print(head)

Head(
  (key): Linear(in_features=64, out_features=16, bias=False)
  (query): Linear(in_features=64, out_features=16, bias=False)
  (value): Linear(in_features=64, out_features=16, bias=False)
  (dropout): Dropout(p=0.2, inplace=False)
)


In [318]:
xb, yb = get_batch("train")

token_embeddings = token_embedding_table(xb)

position_embeddings = position_embedding_table(
    torch.arange(xb.shape[1])
)

x = token_embeddings + position_embeddings

out = head(x)

print("Input shape:")
print(x.shape)

print("\nOutput shape:")
print(out.shape)

Input shape:
torch.Size([16, 32, 64])

Output shape:
torch.Size([16, 32, 16])


## Inspect the Output

Every token now contains a contextualized representation derived from attention.

The output vectors no longer depend solely on the token itself.

Instead, each token can incorporate information from relevant previous tokens.

In [319]:
print("First token output:")
print(out[0, 0])

print("\nOutput vector size:")
print(len(out[0, 0]))

First token output:
tensor([ 0.1369, -1.1249,  0.3891, -0.9840,  0.7794,  0.3048, -0.2481, -0.4107,
        -0.7274,  1.5197,  1.8233, -0.8404, -0.6060, -0.3118,  0.3156, -0.3619],
       grad_fn=<SelectBackward0>)

Output vector size:
16


## Why Multiple Heads?

A single attention head only learns one perspective.

One head may learn:

- grammar

Another may learn:

- punctuation

Another may learn:

- long-range dependencies

Rather than forcing one head to learn everything, GPT uses many heads simultaneously.

This is called:

```text
Multi-Head Attention
```

and is one of the reasons Transformers are so effective.

In the next section we will combine several Heads into a MultiHeadAttention module.

## Multi-Head Attention Architecture

The process is:

```text
Input
  ↓
Head 1
Head 2
Head 3
Head 4
  ↓
Concatenate Outputs
  ↓
Linear Projection
  ↓
Dropout
  ↓
Final Output
```

Each attention head learns its own representation of the input sequence.

The outputs from all heads are concatenated together, producing a larger representation.

The projection layer:

```python
self.proj
```

maps this concatenated representation back into the model's embedding dimension (`n_embd`), allowing information from all heads to be combined into a single representation that can be processed by subsequent Transformer layers.

In [320]:
class MultiHeadAttention(nn.Module):
    """
    Multiple attention heads running in parallel
    """

    def __init__(self, num_heads, head_size):

        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

## Understanding the Components

### ModuleList

```python
self.heads
```

Stores multiple independent attention heads.

Example:

```python
num_heads = 4
```

creates:

```text
Head 1
Head 2
Head 3
Head 4
```

### Projection Layer

```python
self.proj
```

Maps the concatenated outputs back to:

```python
n_embd
```

dimensions.

### Dropout

Helps regularize the model and reduce overfitting.

In [321]:
def forward(self, x):

    out = torch.cat(
        [h(x) for h in self.heads],
        dim=-1
    )

    out = self.proj(out)

    out = self.dropout(out)

    return out


MultiHeadAttention.forward = forward

## Why Concatenation?

Assume:

```python
num_heads = 4
head_size = 16
```

Each attention head outputs:

```text
(B, T, 16)
```

Concatenating all four heads produces:

```text
(B, T, 64)
```

which matches:

```python
n_embd = 64
```

This allows each head to specialize while preserving the original embedding dimension.


## Instantiate Multi-Head Attention

We now create a Multi-Head Attention layer using:

```python
4 heads
16 dimensions per head
```

which produces:

```python
4 × 16 = 64
```

output dimensions.

In [322]:
num_heads = 4
head_size = n_embd // num_heads

multi_head_attn = MultiHeadAttention(
    num_heads,
    head_size
)

print(multi_head_attn)

MultiHeadAttention(
  (heads): ModuleList(
    (0-3): 4 x Head(
      (key): Linear(in_features=64, out_features=16, bias=False)
      (query): Linear(in_features=64, out_features=16, bias=False)
      (value): Linear(in_features=64, out_features=16, bias=False)
      (dropout): Dropout(p=0.2, inplace=False)
    )
  )
  (proj): Linear(in_features=64, out_features=64, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


## Test the Module

The output should preserve the original embedding dimension:

```text
(B, T, C)
```

Input:

```text
(16, 32, 64)
```

Output:

```text
(16, 32, 64)
```

In [323]:
xb, yb = get_batch("train")

token_embeddings = token_embedding_table(xb)

position_embeddings = position_embedding_table(
    torch.arange(xb.shape[1])
)

x = token_embeddings + position_embeddings

out = multi_head_attn(x)

print("Input Shape:")
print(x.shape)

print("\nOutput Shape:")
print(out.shape)

Input Shape:
torch.Size([16, 32, 64])

Output Shape:
torch.Size([16, 32, 64])


## Verify Individual Head Outputs

Each attention head contributes one part of the final representation.

This demonstrates how information is distributed across multiple learning pathways.

In [324]:
for i, head in enumerate(multi_head_attn.heads):

    head_out = head(x)

    print(
        f"Head {i+1} Output Shape:",
        head_out.shape
    )

Head 1 Output Shape: torch.Size([16, 32, 16])
Head 2 Output Shape: torch.Size([16, 32, 16])
Head 3 Output Shape: torch.Size([16, 32, 16])
Head 4 Output Shape: torch.Size([16, 32, 16])


## Why Multi-Head Attention Works

Different attention heads can learn to focus on different types of patterns in the data.

For example, during training one head may learn to track grammatical relationships, while another may learn long-range dependencies or punctuation structure.

Because each head learns independently, the model can analyze the same sequence from multiple perspectives simultaneously.

This diversity of learned representations is one of the key reasons why Transformers are so effective at understanding context.

In [325]:
total_parameters = sum(
    p.numel()
    for p in multi_head_attn.parameters()
)

print(
    f"Multi-Head Attention Parameters: "
    f"{total_parameters:,}"
)


Multi-Head Attention Parameters: 16,448


## Understanding Model Complexity

The number of trainable parameters is an important measure of model complexity.

In this section, we calculated the total number of parameters contained within the Multi-Head Attention module.

These parameters include:

- Query projections
- Key projections
- Value projections
- Output projection layer

As the number of attention heads, embedding dimensions, or Transformer layers increases, the total parameter count grows rapidly.

Monitoring parameter counts helps us understand:

- Memory requirements
- Training speed
- Computational cost
- Model capacity

For large language models, parameter counts often range from millions to billions of parameters.

Our implementation intentionally remains small enough to train on a CPU while still demonstrating the core concepts behind GPT architectures.

# 14. Feed Forward Network

The attention mechanism allows tokens to gather information from other tokens in the sequence.

However, attention alone is not sufficient.

After collecting contextual information, each token representation is passed through a small neural network known as a Feed Forward Network (FFN).

The FFN allows the model to:

- Learn more complex representations
- Transform contextual information
- Introduce non-linearity
- Increase model capacity

Unlike attention, which operates across multiple tokens, the Feed Forward Network processes each token independently.

Attention answers:

> What information should I gather?

The Feed Forward Network answers:

> What should I do with that information?

In [326]:
class FeedForward(nn.Module):
    """
    Simple Feed Forward Network
    """

    def __init__(self, n_embd):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

## Why Expand the Embedding Dimension?

Notice the first layer:

```python
nn.Linear(n_embd, 4 * n_embd)
```

We temporarily increase the embedding size.

For example:

```text
64 → 256 → 64
```

This gives the network additional representational capacity before compressing information back into the original embedding dimension.

This design comes directly from the original Transformer architecture.

In [327]:
def forward(self, x):
    return self.net(x)

FeedForward.forward = forward

## Instantiate the Feed Forward Network

The FFN processes each token independently.

Input shape:

```text
(B, T, C)
```

Output shape:

```text
(B, T, C)
```

The dimensions remain unchanged.

In [328]:
ffwd = FeedForward(n_embd)

print(ffwd)

FeedForward(
  (net): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=64, bias=True)
    (3): Dropout(p=0.2, inplace=False)
  )
)


In [329]:
out_ffwd = ffwd(out)

print("Input Shape:")
print(out.shape)

print("\nOutput Shape:")
print(out_ffwd.shape)

Input Shape:
torch.Size([16, 32, 64])

Output Shape:
torch.Size([16, 32, 64])


## Why We Need Attention and Feed Forward Layers

Attention:

```text
Gather information
```

Feed Forward:

```text
Process information
```

Together:

```text
Attention
     ↓
Feed Forward
```

form the fundamental building block of GPT architectures.

The next step is to combine:

- Multi-Head Attention
- Feed Forward Network
- Residual Connections
- Layer Normalization

into a reusable Transformer Block.

# 15. Transformer Block

We now have the two primary components of a Transformer:

1. Multi-Head Attention
2. Feed Forward Network

Individually, these components are useful, but GPT models combine them into a reusable structure called a **Transformer Block**.

A Transformer Block consists of:

```text
Input
  ↓
Layer Normalization
  ↓
Multi-Head Attention
  ↓
Residual Connection
  ↓
Layer Normalization
  ↓
Feed Forward Network
  ↓
Residual Connection
  ↓
Output
```

Modern GPT architectures stack multiple Transformer Blocks on top of one another to progressively build richer contextual representations.

In our model:

```python
n_layer = 4
```

which means four Transformer Blocks will eventually be stacked together.

## Residual Connections

Deep neural networks can become difficult to train as they grow larger.

Residual connections help by allowing information to flow directly through the network.

Instead of:

```text
Output = Attention(x)
```

we use:

```text
Output = x + Attention(x)
```

This allows the model to:

- Preserve information
- Improve gradient flow
- Train deeper architectures
- Reduce optimization difficulties

Residual connections are one of the key innovations behind modern deep learning architectures.


## Layer Normalization

Layer Normalization stabilizes training by normalizing activations.

Benefits include:

- Faster convergence
- More stable gradients
- Improved training performance
- Better numerical behavior

GPT uses a **Pre-LayerNorm** design:

```text
LayerNorm
    ↓
Attention
```

instead of:

```text
Attention
    ↓
LayerNorm
```

This approach generally trains more reliably.


In [330]:
class Block(nn.Module):
    """
    Transformer Block:
    Communication followed by computation
    """

    def __init__(self, n_embd, n_head):

        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(
            n_embd
        )

        self.ln1 = nn.LayerNorm(
            n_embd
        )

        self.ln2 = nn.LayerNorm(
            n_embd
        )

## Block Components

### Multi-Head Attention

```python
self.sa
```

Allows tokens to exchange information.

### Feed Forward Network

```python
self.ffwd
```

Processes the gathered information independently.

### Layer Normalization

```python
self.ln1
self.ln2
```

Improves training stability.

Together, these components form one complete Transformer Block.

In [331]:
def forward(self, x):

    x = x + self.sa(
        self.ln1(x)
    )

    x = x + self.ffwd(
        self.ln2(x)
    )

    return x


Block.forward = forward

## Understanding the Forward Pass

Step 1:

```text
LayerNorm
    ↓
Multi-Head Attention
    ↓
Residual Connection
```

```python
x = x + self.sa(self.ln1(x))
```

Step 2:

```text
LayerNorm
    ↓
Feed Forward Network
    ↓
Residual Connection
```

```python
x = x + self.ffwd(self.ln2(x))
```

The output retains information from earlier layers while incorporating newly learned representations.

## Instantiate a Transformer Block

We can now create and test a complete Transformer Block.

Input shape:

```text
(B, T, C)
```

Output shape:

```text
(B, T, C)
```

A Transformer Block preserves tensor dimensions while transforming the representation.

In [332]:
block = Block(
    n_embd,
    n_head
)

print(block)

Block(
  (sa): MultiHeadAttention(
    (heads): ModuleList(
      (0-3): 4 x Head(
        (key): Linear(in_features=64, out_features=16, bias=False)
        (query): Linear(in_features=64, out_features=16, bias=False)
        (value): Linear(in_features=64, out_features=16, bias=False)
        (dropout): Dropout(p=0.2, inplace=False)
      )
    )
    (proj): Linear(in_features=64, out_features=64, bias=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (ffwd): FeedForward(
    (net): Sequential(
      (0): Linear(in_features=64, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=64, bias=True)
      (3): Dropout(p=0.2, inplace=False)
    )
  )
  (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
)


In [333]:
xb, yb = get_batch("train")

token_embeddings = token_embedding_table(xb)

position_embeddings = position_embedding_table(
    torch.arange(xb.shape[1])
)

x = token_embeddings + position_embeddings

out = block(x)

print("Input Shape:")
print(x.shape)

print("\nOutput Shape:")
print(out.shape)

Input Shape:
torch.Size([16, 32, 64])

Output Shape:
torch.Size([16, 32, 64])


## Parameter Count

As Transformer Blocks are stacked, the total number of model parameters grows rapidly.

Tracking parameter counts helps us understand:

- Model complexity
- Memory consumption
- Training requirements
- Scaling behavior

Even a relatively small GPT architecture can contain hundreds of thousands or millions of trainable parameters.

In [334]:
total_parameters = sum(
    p.numel()
    for p in block.parameters()
)

print(
    f"Transformer Block Parameters: "
    f"{total_parameters:,}"
)

Transformer Block Parameters: 49,792


## Why Transformer Blocks Matter

The Transformer Block is the fundamental building block of GPT models.

GPT architectures are created by stacking many Transformer Blocks:

```text
Input
  ↓
Block 1
  ↓
Block 2
  ↓
Block 3
  ↓
Block 4
  ↓
Output Layer
```

Each successive block builds more sophisticated contextual representations.

This repeated structure enables GPT models to learn grammar, semantics, long-range dependencies, reasoning patterns, and language structure.

## Section Summary

In this section, we combined the key components of the Transformer architecture into a reusable Transformer Block.

We:

- Integrated Multi-Head Attention
- Integrated the Feed Forward Network
- Added Layer Normalization
- Added Residual Connections
- Preserved input and output dimensions
- Built the fundamental building block used throughout GPT models

A Transformer Block performs two essential operations:

1. Gather contextual information using attention.
2. Transform that information using a feed-forward network.

Modern GPT architectures are formed by stacking many Transformer Blocks together, allowing progressively richer language representations to emerge throughout the network.

At this point, we have implemented all the major components required to construct a complete GPT language model.

# 16. GPT Language Model

We now have all the pieces needed to build a complete GPT architecture.

So far, we have implemented:

- Token Embeddings
- Positional Embeddings
- Self-Attention
- Multi-Head Attention
- Feed Forward Networks
- Transformer Blocks

In this section, we will combine these components into a single GPT language model capable of:

- Learning from text
- Predicting the next token
- Minimizing training loss
- Generating entirely new text

The model architecture will follow the standard GPT pipeline:

```text
Input Tokens
      ↓
Token Embeddings
      ↓
Positional Embeddings
      ↓
Transformer Blocks
      ↓
Layer Normalization
      ↓
Language Modeling Head
      ↓
Vocabulary Probabilities
```

This will be the first fully trainable language model in our project.

In [335]:
class GPTLanguageModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head)
                for _ in range(n_layer)
            ]
        )

        self.ln_f = nn.LayerNorm(
            n_embd
        )

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

## GPT Architecture Components

### Token Embeddings

```python
self.token_embedding_table
```

Converts token IDs into dense vector representations.

### Positional Embeddings

```python
self.position_embedding_table
```

Provides sequence-order information.

### Transformer Blocks

```python
self.blocks
```

A stack of Transformer layers that refine token representations.

### Final Layer Normalization

```python
self.ln_f
```

Stabilizes the final activations before prediction.

### Language Modeling Head

```python
self.lm_head
```

Projects embeddings into vocabulary logits representing the probability of every possible next token.

In [336]:
def forward(self, idx, targets=None):

    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx)

    pos_emb = self.position_embedding_table(
        torch.arange(T, device=device)
    )

    x = tok_emb + pos_emb

    x = self.blocks(x)

    x = self.ln_f(x)

    logits = self.lm_head(x)

    if targets is None:

        loss = None

    else:

        B, T, C = logits.shape

        logits = logits.view(B * T, C)

        targets = targets.view(B * T)

        loss = F.cross_entropy(
            logits,
            targets
        )

    return logits, loss


GPTLanguageModel.forward = forward


## Testing the GPT Language Model

Before training, we verify that the architecture executes successfully.

At this stage:

- The model is randomly initialized.
- Predictions are meaningless.
- We are only checking tensor shapes and loss calculation.

A successful forward pass confirms that all model components are connected correctly.

In [337]:
model = GPTLanguageModel()

xb, yb = get_batch("train")

logits, loss = model(xb, yb)

print("Logits Shape:")
print(logits.shape)

print("\nLoss:")
print(loss)


Logits Shape:
torch.Size([512, 65])

Loss:
tensor(4.3555, grad_fn=<NllLossBackward0>)


## Counting Parameters

In [338]:
total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Total GPT Parameters: {total_parameters:,}"
)

Total GPT Parameters: 209,729


## Generate Method

In [339]:
def generate(self, idx, max_new_tokens):
    ...

## Text Generation

A language model becomes useful when it can generate new text.

Generation works by repeatedly:

1. Predicting the next token.
2. Sampling from the probability distribution.
3. Appending the new token to the sequence.
4. Repeating the process.

Because the model has not yet been trained, the generated text will appear random.

After training, coherent Shakespeare-style language should emerge.

In [340]:
def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, loss = self(idx_cond)

        logits = logits[:, -1, :]

        probs = F.softmax(
            logits,
            dim=-1)

        idx_next = torch.multinomial(
            probs,
            num_samples=1)

        idx = torch.cat(
            (idx, idx_next),
            dim=1)

    return idx


GPTLanguageModel.generate = generate

## Test Untrained Generation

The model has not yet learned any language patterns.

The purpose of this test is simply to verify that text generation works end-to-end.

The generated output should look random at this stage.

In [341]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long)

generated = model.generate(
    context,
    max_new_tokens=100)

print(decode(generated[0].tolist()))


ypqj!!qmCC
!byJ;eVXnqg'?oSzohMAhhRKB,qfJvzl$a
HWX'Rk? RM&j;nnQf:X-a$eF -:,Kazkhn'BFWJBxVJ$lbjhRGhD':


# 17. Training the GPT Model

The model architecture is now complete.

However, the model currently contains randomly initialized weights and has not yet learned any language patterns.

Training allows the model to gradually improve by repeatedly:

1. Generating predictions.
2. Comparing predictions to the correct targets.
3. Calculating loss.
4. Updating parameters using gradient descent.

Over time, the training loss decreases and the model begins to learn the structure of Shakespearean language.

In this section we will:

- Create an optimizer
- Monitor training loss
- Monitor validation loss
- Train the model
- Evaluate performance

## Initialize the Model

Before training, we move the model to the selected device and create an optimizer.

We will use the AdamW optimizer, which is commonly used for training Transformer architectures because it provides stable optimization and includes weight decay regularization.

In [342]:
model = GPTLanguageModel()

model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate)

print(f"Model Parameters: "f"{sum(p.numel() for p in model.parameters()):,}")

Model Parameters: 209,729


## Estimate Loss Function

Training loss alone is not sufficient.

We also monitor validation loss using unseen data.

Tracking both losses helps us determine:

- Whether the model is learning
- Whether overfitting is occurring
- Whether training should continue

The estimate_loss function evaluates model performance without updating weights.

In [343]:
@torch.no_grad()
def estimate_loss():

    out = {}

    model.eval()

    for split in ["train", "val"]:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):

            X, Y = get_batch(split)

            X = X.to(device)
            Y = Y.to(device)

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

## Training Loop

The training loop repeatedly:

1. Samples a batch.
2. Performs a forward pass.
3. Computes the loss.
4. Calculates gradients.
5. Updates model parameters.

Training progress is displayed periodically using both training and validation loss.

In [344]:
for iter in range(max_iters):

    if iter % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Step {iter}: "
            f"Train Loss {losses['train']:.4f}, "
            f"Val Loss {losses['val']:.4f}"
        )

    xb, yb = get_batch("train")

    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

Step 0: Train Loss 4.3727, Val Loss 4.3608
Step 100: Train Loss 2.6699, Val Loss 2.6771
Step 200: Train Loss 2.5294, Val Loss 2.5405
Step 300: Train Loss 2.4392, Val Loss 2.4490
Step 400: Train Loss 2.3840, Val Loss 2.3870
Step 500: Train Loss 2.3247, Val Loss 2.3412
Step 600: Train Loss 2.2963, Val Loss 2.3071
Step 700: Train Loss 2.2466, Val Loss 2.2667
Step 800: Train Loss 2.2375, Val Loss 2.2275
Step 900: Train Loss 2.1836, Val Loss 2.1903
Step 1000: Train Loss 2.1577, Val Loss 2.1867
Step 1100: Train Loss 2.1266, Val Loss 2.1754
Step 1200: Train Loss 2.1081, Val Loss 2.1426
Step 1300: Train Loss 2.0882, Val Loss 2.1129
Step 1400: Train Loss 2.0795, Val Loss 2.1092
Step 1500: Train Loss 2.0445, Val Loss 2.0950
Step 1600: Train Loss 2.0375, Val Loss 2.0899
Step 1700: Train Loss 2.0212, Val Loss 2.0702
Step 1800: Train Loss 2.0046, Val Loss 2.0772
Step 1900: Train Loss 1.9818, Val Loss 2.0516
Step 2000: Train Loss 1.9766, Val Loss 2.0341
Step 2100: Train Loss 1.9648, Val Loss 2.0286


## Expected Training Behavior

Initially, the loss will be relatively high:

```text
4.0+
```

As training progresses:

```text
4.0
↓
3.5
↓
3.0
↓
2.5
```

the model begins learning meaningful language patterns.

Training time depends on:

- CPU speed
- Batch size
- Number of layers
- Dataset size

On a CPU-only machine, several thousand iterations may take a few minutes.

# 18. Saving the Trained Model

Training GPT models can take considerable time.

To avoid retraining the model every time we want to generate text, we save the learned parameters to disk.

PyTorch stores the model weights using:

```python
state_dict()
```

These saved weights can later be loaded to resume training or generate text.

In [345]:
import os

os.makedirs("../models", exist_ok=True)

MODEL_PATH = "../models/gpt_shakespeare.pth"

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print(
    f"Model saved to:\n{MODEL_PATH}"
)

Model saved to:
../models/gpt_shakespeare.pth


## Verify Saved Model

After saving, we verify that the model file exists.

This confirms that training artifacts have been successfully written to disk.

In [346]:
import os

print(
    "Model Exists:",
    os.path.exists(MODEL_PATH)
)


Model Exists: True


# 19. Generate Text Using the Trained GPT Model

Now that the model has been trained, we can use it to generate Shakespeare-like text.

Unlike the random text produced before training, the model should now demonstrate:

- Words
- Names
- Sentence structure
- Shakespearean patterns
- Dramatic dialogue


In [347]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated = model.generate(
    context,
    max_new_tokens=500
)

print(
    decode(
        generated[0].tolist()
    )
)


Lece in weop as welk a clids me didal both islow are' the for me, a
thyse,
Biant know, but for the find wunt Lyoy,
That an are tell that thrishconatore fon tere pesd farle
To tis, ore held eyem'd no thoodfore your ain,
Pome and arehe heely arloboud'd in there.

CORIANCE:

CLA:
Wry chevere suees
Hougar, will forrim?

SICLO:
Sters, lother plazciour, many dead
Tnor hoce not devirgate, work, bece.

MONEShall And I Tintipp:
Ase, feer or wybles; of sainst he
Girrice.
GLech and Rone's me suo-will hing 


## Evaluating Generated Text

At this stage we are looking for evidence that the model has learned:

- English word structure
- Character names
- Basic grammar
- Shakespearean formatting

Although the generated text may still contain mistakes, it should be noticeably more coherent than the completely random output produced before training.

This demonstrates that the Transformer has successfully learned patterns from the training corpus.

# 18. Save & Reload Models

Training a Transformer model can take a significant amount of time.

Rather than retraining the model every time we want to generate text, we save the learned model parameters to disk.

PyTorch stores model weights using:

```python
state_dict()
```

Saving and loading models is a critical production skill because it allows:

- Reuse of trained models
- Reproducible experiments
- Model deployment
- Continued training from checkpoints

In this section, we save the trained GPT model and verify that it can be loaded successfully.

In [348]:
import os

MODEL_PATH = "../models/gpt_shakespeare.pth"

os.makedirs("../models", exist_ok=True)

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print(f"Model saved to: {MODEL_PATH}")


Model saved to: ../models/gpt_shakespeare.pth


## Verify the Saved Model

After saving the model, we verify that the file exists on disk.

This confirms that the training results have been successfully preserved.

In [349]:
import os

print(
    "Model Exists:",
    os.path.exists(MODEL_PATH)
)

Model Exists: True


## Reload the Saved Model

A saved model can be loaded into a new instance of the same architecture.

This demonstrates how trained models can be reused without repeating the training process.

In [350]:
loaded_model = GPTLanguageModel()

loaded_model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

loaded_model.eval()

print("Model loaded successfully.")

Model loaded successfully.


## Generate Text from the Reloaded Model

To verify that loading worked correctly, we generate text using the reloaded model.

The generated text should be similar in quality to the original trained model.

In [351]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long
)

generated = loaded_model.generate(
    context,
    max_new_tokens=300
)

print(
    decode(
        generated[0].tolist()
    )
)


I the, he shall wing treated sweep ssabt's is vity,
Gricy, Sare; a grefall is subtern must agafe,
Where to herem'd fatter faoly, to is my foe.

YOLIONUS:
Edward, tast At I hy.

Foret my like, virs!
Rawed have surn,
I he boy be, exm tall be not you stave.

Sen my the me, my let, the fort Creat is now


# 19. Evaluate Generation Quality

Evaluating language models can be challenging because text quality is subjective.

For this project, we perform a qualitative evaluation by examining generated samples.

We focus on several questions:

- Does the model generate recognizable words?
- Does it learn punctuation?
- Does it learn dialogue formatting?
- Does the output resemble Shakespearean text?
- Does the output show signs of grammatical structure?

These observations help us determine whether the model has successfully learned patterns from the training corpus.

In [352]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long
)

sample = loaded_model.generate(
    context,
    max_new_tokens=1000
)

generated_text = decode(
    sample[0].tolist()
)

print(generated_text)


Yet by casurns in I do dir lay's and, there of yignd:
Ster'd? know of them shame with how,
To king the marce their deem, Pain's aft,
That cartion way: unk at lodg come be the than fueet meetfull and fatcy.

LUCIO:
I in they If mrore, for at to that dence of cales;
Let evers sin the he flaalt old sir,
you of ruservicent do would you:
Who, shall auld shold, he seep tood, my could beok accent
Which tulfore a can spertour but.

LAUCENSTARENTIO:
I the ever were to mook.

KING RICHARD II:
A that by thou?

PELO:
'LOUCEY:
Go; ever's be shoubt you'd pastly bee.

KING RICHARD III:
O, all it be?

CLUDIUS:
We the a sweall, the I extriang are
ever deverse min therefully
Uuon would, ever eye pram for fame for thee!

Clanyse my loves, and it to my a goped wouth sonst sta from you slaid:
I do no me this he me be; a my such grether my live todle,
At choldilses mirry went him, let lurse ewnid but we day, nue!
Thy Vell fall besurt, diese thy crop thy hese old.

TARENTIO:
And the
dentleop thit be and you

## Evaluation Results

The trained model demonstrates several characteristics of successful learning:

### Learned Successfully

- Character names appear
- Dialogue structure appears
- English-like words appear
- Shakespeare-style formatting emerges
- Punctuation usage is learned

### Areas for Improvement

- Grammar remains inconsistent
- Many words are invented
- Long-range coherence is limited
- Character interactions are often nonsensical

These limitations are expected because the model is relatively small and was trained for only a limited number of optimization steps.

In [353]:
print("Final Training Loss Approximation:")
print("Train Loss: ~1.77")

print("\nFinal Validation Loss Approximation:")
print("Validation Loss: ~1.91")

Final Training Loss Approximation:
Train Loss: ~1.77

Final Validation Loss Approximation:
Validation Loss: ~1.91


## Interpretation

The reduction in loss from approximately:

```text
4.37
```

to:

```text
1.77
```

indicates that the model successfully learned meaningful patterns from the dataset.

Although the generated text is not yet fully coherent, the model has clearly moved beyond random character generation and learned aspects of language structure.

# 20. Future Improvements & Next Steps

This project demonstrated how a GPT-style language model can be built entirely from scratch using PyTorch.

While the current model is intentionally small enough to train on a CPU, there are many opportunities for further development.

## Model Improvements

Possible architectural improvements include:

### Larger Embedding Dimensions

Current:

```python
n_embd = 64
```

Possible upgrade:

```python
n_embd = 128
```

or

```python
n_embd = 256
```

### More Transformer Layers

Current:

```python
n_layer = 4
```

Possible upgrade:

```python
n_layer = 6
```

or

```python
n_layer = 8
```

### Larger Context Windows

Current:

```python
block_size = 32
```

Possible upgrade:

```python
block_size = 64
```

or

```python
block_size = 128
```

Larger context windows allow the model to capture longer-range dependencies.

## Training Improvements

Potential training improvements include:

- More iterations
- Learning rate scheduling
- Weight decay tuning
- Gradient clipping
- Better evaluation metrics

Longer training generally produces more coherent generated text.

## Hardware Improvements

Training on a GPU would significantly increase:

- Training speed
- Model size
- Context window size
- Experimentation capability

Examples include:

- NVIDIA RTX GPUs
- Cloud-based GPU environments
- Managed machine learning services

## Domain Adaptation: Logistics GPT

One of the most exciting next steps is fine-tuning the model on logistics and supply-chain data.

Potential sources include:

- Incoterms documentation
- Bills of lading
- Commercial invoices
- Customs documentation
- Shipping updates
- Trade compliance references

A domain-specific GPT could generate text related to:

- Supply-chain management
- International trade
- Customs compliance
- Freight forwarding operations

This would create a unique portfolio project that combines machine learning with logistics expertise.

## Project Conclusion

In this project, we successfully:

✅ Loaded and processed a text corpus

✅ Created tokenization and decoding pipelines

✅ Built embeddings and positional encodings

✅ Implemented self-attention

✅ Implemented multi-head attention

✅ Built Transformer blocks

✅ Constructed a GPT language model

✅ Trained the model on Shakespeare text

✅ Generated novel text

✅ Saved and reloaded trained model checkpoints

This project provides a complete end-to-end introduction to GPT architectures and demonstrates the core ideas behind modern Large Language Models.